# Reproducible pipeline: data preparation, model training, evaluation

This notebook is the single executable entry point for the thesis bi-encoder pipeline. It rebuilds derived data from raw sources, trains `cointegrated/rubert-tiny2`, evaluates binary pair separation, and evaluates query-level ranking metrics on the full test split.

Run with nbconvert:

```bash
python -m jupyter nbconvert --to notebook --execute experiments/notebooks/pipeline_prepare_model.ipynb --output pipeline_prepare_model.executed.ipynb --ExecutePreprocessor.timeout=-1
```

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

ROOT = Path.cwd()
if not (ROOT / 'experiments' / 'scripts').exists():
    ROOT = Path.cwd().parents[0]

DATA_DIR = ROOT / 'data'
SCRIPTS_DIR = ROOT / 'experiments' / 'scripts'
RESULTS_DIR = ROOT / 'experiments' / 'results'
MODELS_DIR = ROOT / 'experiments' / 'models'
MODEL_OUTPUT = MODELS_DIR / 'bi_encoder_rubert_tiny2'

BASE_MODEL = os.environ.get('BASE_MODEL', 'cointegrated/rubert-tiny2')
SAMPLE_SIZE = int(os.environ.get('SAMPLE_SIZE', '10000'))
BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '8'))
EPOCHS = int(os.environ.get('EPOCHS', '3'))
WARMUP_STEPS = int(os.environ.get('WARMUP_STEPS', '100'))

RUN_DATA_PREP = os.environ.get('RUN_DATA_PREP', '1') == '1'
RUN_TRAINING = os.environ.get('RUN_TRAINING', '1') == '1'
RUN_EVALUATION = os.environ.get('RUN_EVALUATION', '1') == '1'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT=', ROOT)
print('DATA_DIR=', DATA_DIR)
print('BASE_MODEL=', BASE_MODEL)
print('MODEL_OUTPUT=', MODEL_OUTPUT)
print('RUN_DATA_PREP=', RUN_DATA_PREP)
print('RUN_TRAINING=', RUN_TRAINING)
print('RUN_EVALUATION=', RUN_EVALUATION)

## Helper for reproducible command execution

All subprocess output is streamed to the notebook. A non-zero exit code stops the notebook execution.

In [ ]:
def run_cmd(args, cwd=ROOT, env=None):
    started = time.time()
    print('\n$ ' + ' '.join(str(a) for a in args))
    completed = subprocess.run([str(a) for a in args], cwd=str(cwd), env=env)
    elapsed = time.time() - started
    print(f'EXIT_CODE={completed.returncode} elapsed_seconds={elapsed:.3f}')
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {args}')
    return elapsed

## 1. Data preparation

The preparation stage rebuilds derived files from raw matched HR events: sample, clean, normalized, feature, unified, and split files. `02_profile_dataset.py` is not required for model training and is intentionally not part of the blocking pipeline.

In [ ]:
if RUN_DATA_PREP:
    prep_commands = [
        [sys.executable, SCRIPTS_DIR / '01_sample_and_profile_data.py', '--sample_size', SAMPLE_SIZE],
        [sys.executable, SCRIPTS_DIR / '03_clean_and_mask_data.py'],
        [sys.executable, SCRIPTS_DIR / '05_text_normalization.py'],
        [sys.executable, SCRIPTS_DIR / '06_entity_extraction.py'],
        [sys.executable, SCRIPTS_DIR / '07_feature_formatting.py'],
        [sys.executable, SCRIPTS_DIR / '04_build_dataset_splits.py'],
    ]
    for cmd in prep_commands:
        run_cmd(cmd)
else:
    print('Skipping data preparation because RUN_DATA_PREP=0')

## 2. Training

Training uses only `cointegrated/rubert-tiny2` by default, `batch_size=8`, and `epochs=3`. The training script rebuilds `data/splits/model_features.db` through `08_model_dataset.py`.

In [ ]:
if RUN_TRAINING:
    train_cmd = [
        sys.executable,
        SCRIPTS_DIR / '09_train_biencoder.py',
        '--model_name', BASE_MODEL,
        '--data_dir', DATA_DIR,
        '--output_dir', MODELS_DIR,
        '--batch_size', BATCH_SIZE,
        '--epochs', EPOCHS,
        '--warmup_steps', WARMUP_STEPS,
    ]
    train_elapsed = run_cmd(train_cmd)
    (RESULTS_DIR / 'notebook_training_runtime.json').write_text(
        json.dumps({'train_wall_time_seconds': train_elapsed}, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
else:
    print('Skipping training because RUN_TRAINING=0')

if not MODEL_OUTPUT.exists():
    raise FileNotFoundError(f'Model output path does not exist: {MODEL_OUTPUT}')

## 3. Binary pair evaluation

This cell compares the base model with the fine-tuned model on the current `val.tsv` and `test.tsv`. It saves exact raw values to `experiments/results/notebook_binary_eval.json`.

In [ ]:
if RUN_EVALUATION:
    import numpy as np
    import pandas as pd
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics import accuracy_score, average_precision_score, matthews_corrcoef, precision_recall_fscore_support

    def read_tsv(path):
        return pd.read_csv(path, sep='\t', encoding='utf-8-sig')

    resumes = read_tsv(DATA_DIR / 'unified' / 'resumes_unified.tsv')
    vacancies = read_tsv(DATA_DIR / 'unified' / 'vacancies_unified.tsv')
    resume_text = dict(zip(resumes['id'].astype(str), resumes['model_input'].fillna('').astype(str)))
    vacancy_text = dict(zip(vacancies['id'].astype(str), vacancies['model_input'].fillna('').astype(str)))

    def cosine_rows(a, b):
        a = a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-12, None)
        b = b / np.clip(np.linalg.norm(b, axis=1, keepdims=True), 1e-12, None)
        return np.sum(a * b, axis=1)

    def best_threshold_metrics(y_true, scores):
        best = None
        for threshold in np.unique(scores):
            y_pred = (scores >= threshold).astype(int)
            precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
            accuracy = accuracy_score(y_true, y_pred)
            mcc = matthews_corrcoef(y_true, y_pred) if len(set(y_pred)) > 1 and len(set(y_true)) > 1 else 0.0
            row = {
                'threshold': float(threshold),
                'accuracy': float(accuracy),
                'precision': float(precision),
                'recall': float(recall),
                'f1': float(f1),
                'mcc': float(mcc),
            }
            if best is None or (row['f1'], row['accuracy']) > (best['f1'], best['accuracy']):
                best = row
        return best

    def eval_model(model_name, model_path):
        model = SentenceTransformer(str(model_path), device='cpu')
        output = {}
        for split_name in ['val', 'test']:
            pairs = read_tsv(DATA_DIR / 'splits' / f'{split_name}.tsv')
            labels = pairs['label'].astype(int).to_numpy()
            s1 = [resume_text[str(x)] for x in pairs['resume_id'].astype(str)]
            s2 = [vacancy_text[str(x)] for x in pairs['vacancy_id'].astype(str)]
            emb1 = model.encode(s1, batch_size=32, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=False)
            emb2 = model.encode(s2, batch_size=32, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=False)
            scores = cosine_rows(emb1, emb2)
            output[split_name] = {
                'rows': int(len(labels)),
                'positives': int(labels.sum()),
                'negatives': int((1 - labels).sum()),
                'average_precision': float(average_precision_score(labels, scores)),
                'best_f1_metrics': best_threshold_metrics(labels, scores),
                'score_pos_mean': float(np.mean(scores[labels == 1])),
                'score_neg_mean': float(np.mean(scores[labels == 0])),
            }
        return output

    binary_results = {
        'baseline': eval_model('baseline', BASE_MODEL),
        'fine_tuned': eval_model('fine_tuned', MODEL_OUTPUT),
    }
    for split_name in ['val', 'test']:
        b = binary_results['baseline'][split_name]
        f = binary_results['fine_tuned'][split_name]
        binary_results[f'delta_{split_name}'] = {
            'average_precision': f['average_precision'] - b['average_precision'],
            'best_f1': f['best_f1_metrics']['f1'] - b['best_f1_metrics']['f1'],
            'best_accuracy': f['best_f1_metrics']['accuracy'] - b['best_f1_metrics']['accuracy'],
            'best_mcc': f['best_f1_metrics']['mcc'] - b['best_f1_metrics']['mcc'],
        }

    out_path = RESULTS_DIR / 'notebook_binary_eval.json'
    out_path.write_text(json.dumps(binary_results, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(binary_results, ensure_ascii=False, indent=2))
    print('saved=', out_path)
else:
    print('Skipping binary evaluation because RUN_EVALUATION=0')

## 4. Full test split ranking evaluation

This cell ranks resumes within each `vacancy_id` group from the real `data/splits/test.tsv`. NDCG is calculated only for groups where it is mathematically defined: at least two candidates and at least one positive label.

In [ ]:
if RUN_EVALUATION:
    import numpy as np
    import pandas as pd
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics import ndcg_score

    K_VALUES = [1, 3, 5, 10]

    def read_tsv(path):
        return pd.read_csv(path, sep='\t', encoding='utf-8-sig')

    def precision_at_k(labels, k):
        top = labels[:min(k, len(labels))]
        return float(np.sum(top) / len(top)) if len(top) else 0.0

    def recall_at_k(labels, total_relevant, k):
        if total_relevant <= 0:
            return None
        top = labels[:min(k, len(labels))]
        return float(np.sum(top) / total_relevant)

    def mrr(labels):
        for index, rel in enumerate(labels, start=1):
            if int(rel) == 1:
                return float(1.0 / index)
        return 0.0

    def ndcg_at_k(labels, scores, k):
        if len(labels) < 2 or np.sum(labels) <= 0:
            return None
        return float(ndcg_score(np.asarray([labels]), np.asarray([scores]), k=min(k, len(labels))))

    def mean_present(values):
        vals = [v for v in values if v is not None]
        return float(np.mean(vals)) if vals else None

    test = read_tsv(DATA_DIR / 'splits' / 'test.tsv')
    resumes = read_tsv(DATA_DIR / 'unified' / 'resumes_unified.tsv')
    vacancies = read_tsv(DATA_DIR / 'unified' / 'vacancies_unified.tsv')
    resume_text = dict(zip(resumes['id'].astype(str), resumes['model_input'].fillna('').astype(str)))
    vacancy_text = dict(zip(vacancies['id'].astype(str), vacancies['model_input'].fillna('').astype(str)))

    missing_resumes = int((~test['resume_id'].astype(str).isin(resume_text)).sum())
    missing_vacancies = int((~test['vacancy_id'].astype(str).isin(vacancy_text)).sum())
    if missing_resumes or missing_vacancies:
        raise RuntimeError(f'Missing split IDs: resumes={missing_resumes}, vacancies={missing_vacancies}')

    model = SentenceTransformer(str(MODEL_OUTPUT), device='cpu')
    unique_resume_ids = test['resume_id'].astype(str).drop_duplicates().tolist()
    unique_vacancy_ids = test['vacancy_id'].astype(str).drop_duplicates().tolist()
    resume_embeddings = model.encode([resume_text[i] for i in unique_resume_ids], batch_size=32, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)
    vacancy_embeddings = model.encode([vacancy_text[i] for i in unique_vacancy_ids], batch_size=32, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)
    resume_emb = dict(zip(unique_resume_ids, resume_embeddings))
    vacancy_emb = dict(zip(unique_vacancy_ids, vacancy_embeddings))

    test = test.copy()
    test['score'] = [float(np.dot(resume_emb[str(row.resume_id)], vacancy_emb[str(row.vacancy_id)])) for row in test.itertuples(index=False)]

    group_metrics = []
    for vacancy_id, group in test.groupby('vacancy_id', sort=False):
        sorted_group = group.sort_values('score', ascending=False)
        labels = sorted_group['label'].astype(int).to_numpy()
        scores = sorted_group['score'].to_numpy()
        total_relevant = int(labels.sum())
        item = {
            'vacancy_id': str(vacancy_id),
            'n_candidates': int(len(labels)),
            'n_relevant': total_relevant,
            'mrr': mrr(labels) if total_relevant > 0 else None,
        }
        for k in K_VALUES:
            item[f'ndcg@{k}'] = ndcg_at_k(labels, scores, k)
            item[f'precision@{k}'] = precision_at_k(labels, k)
            item[f'recall@{k}'] = recall_at_k(labels, total_relevant, k)
        group_metrics.append(item)

    eval_groups = [g for g in group_metrics if g['n_relevant'] > 0]
    ndcg_eligible_groups = [g for g in group_metrics if g['n_relevant'] > 0 and g['n_candidates'] >= 2]
    macro_positive_queries = {'queries': len(eval_groups), 'ndcg_eligible_queries': len(ndcg_eligible_groups)}
    for k in K_VALUES:
        macro_positive_queries[f'ndcg@{k}'] = mean_present([g[f'ndcg@{k}'] for g in eval_groups])
        macro_positive_queries[f'precision@{k}'] = mean_present([g[f'precision@{k}'] for g in eval_groups])
        macro_positive_queries[f'recall@{k}'] = mean_present([g[f'recall@{k}'] for g in eval_groups])
    macro_positive_queries['mrr'] = mean_present([g['mrr'] for g in eval_groups])

    ranking_results = {
        'test_rows': int(len(test)),
        'unique_resumes': int(test['resume_id'].nunique()),
        'unique_vacancies': int(test['vacancy_id'].nunique()),
        'total_positives': int(test['label'].sum()),
        'total_negatives': int((test['label'] == 0).sum()),
        'missing_resumes': missing_resumes,
        'missing_vacancies': missing_vacancies,
        'macro_positive_queries': macro_positive_queries,
        'group_metrics': group_metrics,
    }
    out_path = RESULTS_DIR / 'notebook_full_test_ranking_metrics.json'
    out_path.write_text(json.dumps(ranking_results, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({k: v for k, v in ranking_results.items() if k != 'group_metrics'}, ensure_ascii=False, indent=2))
    print('saved=', out_path)
else:
    print('Skipping ranking evaluation because RUN_EVALUATION=0')

## 5. API export command

The API server now requires `MODEL_PATH`. The model path below is the path produced by this notebook.

In [ ]:
print('PowerShell:')
print(f"$env:MODEL_PATH='{MODEL_OUTPUT}'; .\\.venv\\Scripts\\python.exe experiments\\scripts\\11_api_server.py")
print('\nBash:')
print(f"MODEL_PATH='{MODEL_OUTPUT}' python experiments/scripts/11_api_server.py")